# Lista 11

In [1]:
import numpy as np
import time

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

SEED = 42

X, y = make_classification(
    n_samples=200,
    n_features=5000,
    n_informative=30,     # kilka prawdziwie istotnych genów
    n_redundant=30,       # trochę skorelowanych
    n_repeated=0,
    n_classes=2,
    flip_y=0.03,
    class_sep=1.0,
    random_state=SEED,
    shuffle=True
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)


In [2]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=SEED,
    n_jobs=-1
)

t0 = time.perf_counter()
rf.fit(X_train, y_train)
train_time = time.perf_counter() - t0

acc_train = rf.score(X_train, y_train)
acc_test = rf.score(X_test, y_test)

importances = rf.feature_importances_
nonzero = np.sum(importances > 0)

print("RF all features")
print("train_acc:", acc_train)
print("test_acc :", acc_test)
print("train_time_s:", train_time)
print("nonzero_importances:", nonzero, "out of", X.shape[1])


RF all features
train_acc: 1.0
test_acc : 0.5166666666666667
train_time_s: 0.2308099580113776
nonzero_importances: 2597 out of 5000


In [3]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def eval_kbest(k):
    pipe = Pipeline([
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("clf", LogisticRegression(max_iter=1000, solver="liblinear"))
    ])
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    fit_t = time.perf_counter() - t0

    tr = accuracy_score(y_train, pipe.predict(X_train))
    te = accuracy_score(y_test, pipe.predict(X_test))
    return tr, te, fit_t

for k in [50, 100, 200]:
    tr, te, fit_t = eval_kbest(k)
    print(f"k={k}: train_acc={tr:.3f}, test_acc={te:.3f}, fit_time_s={fit_t:.3f}")


k=50: train_acc=1.000, test_acc=0.583, fit_time_s=0.008
k=100: train_acc=1.000, test_acc=0.567, fit_time_s=0.008
k=200: train_acc=1.000, test_acc=0.633, fit_time_s=0.008


In [4]:
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler

# 1) KBest(100) jako punkt odniesienia
kbest = SelectKBest(f_classif, k=100).fit(X_train, y_train)
kbest_idx = set(kbest.get_support(indices=True))

# 2) RFE(100) z LogisticRegression
base_est = LogisticRegression(max_iter=2000, solver="liblinear")
rfe = RFE(estimator=base_est, n_features_to_select=100, step=0.2)
t0 = time.perf_counter()
rfe.fit(X_train, y_train)
rfe_time = time.perf_counter() - t0
rfe_idx = set(np.where(rfe.support_)[0])

# 3) L1 (embedded) - wymaga skalowania, solver saga
lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty="l1", solver="saga", C=0.1, max_iter=5000, n_jobs=-1))
])
t0 = time.perf_counter()
lasso.fit(X_train, y_train)
lasso_time = time.perf_counter() - t0
coef = lasso.named_steps["clf"].coef_.ravel()
lasso_idx = set(np.where(np.abs(coef) > 1e-9)[0])  # cechy z niezerowym współczynnikiem

# 4) RF importance top-100 (z części (a) albo ucz ponownie)
rf_top100_idx = set(np.argsort(importances)[::-1][:100])

def jaccard(a, b):
    return len(a & b) / len(a | b)

def acc_on_idx(idx):
    idx = sorted(idx)
    Xtr = X_train[:, idx]
    Xte = X_test[:, idx]
    clf = LogisticRegression(max_iter=2000, solver="liblinear")
    clf.fit(Xtr, y_train)
    return clf.score(Xtr, y_train), clf.score(Xte, y_test)

print("Accuracies (LogReg trained on selected features):")
print("KBest100:", acc_on_idx(kbest_idx))
print("RFE100  :", (rfe.score(X_train, y_train), rfe.score(X_test, y_test)))
print("L1      :", (lasso.score(X_train, y_train), lasso.score(X_test, y_test)), "selected:", len(lasso_idx))
print("RF top100:", acc_on_idx(rf_top100_idx))

print("\nJaccard overlaps:")
pairs = [
    ("kbest", kbest_idx, "rfe", rfe_idx),
    ("kbest", kbest_idx, "lasso", lasso_idx),
    ("kbest", kbest_idx, "rf", rf_top100_idx),
    ("rfe", rfe_idx, "lasso", lasso_idx),
    ("rfe", rfe_idx, "rf", rf_top100_idx),
    ("lasso", lasso_idx, "rf", rf_top100_idx),
]
for a_name, a, b_name, b in pairs:
    print(f"{a_name} vs {b_name}: {jaccard(a,b):.3f}")


Accuracies (LogReg trained on selected features):
KBest100: (1.0, 0.5666666666666667)
RFE100  : (1.0, 0.7333333333333333)
L1      : (0.9785714285714285, 0.5666666666666667) selected: 45
RF top100: (1.0, 0.6333333333333333)

Jaccard overlaps:
kbest vs rfe: 0.274
kbest vs lasso: 0.408
kbest vs rf: 0.183
rfe vs lasso: 0.218
rfe vs rf: 0.093
lasso vs rf: 0.133


In [5]:
from sklearn.neighbors import KNeighborsClassifier

ps = [20, 100, 500, 1000, 5000]

for p in ps:
    n_inf = min(10, max(2, p // 10))
    Xp, yp = make_classification(
        n_samples=200, n_features=p,
        n_informative=n_inf, n_redundant=0,
        n_classes=2, flip_y=0.03, class_sep=1.0,
        random_state=SEED
    )
    Xtr, Xte, ytr, yte = train_test_split(Xp, yp, test_size=0.3, random_state=SEED, stratify=yp)

    knn = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))])
    knn.fit(Xtr, ytr)

    lr = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=2000))])
    lr.fit(Xtr, ytr)

    print(f"p={p:4d} | KNN test={knn.score(Xte,yte):.3f} | LogReg test={lr.score(Xte,yte):.3f}")


p=  20 | KNN test=0.700 | LogReg test=0.883
p= 100 | KNN test=0.633 | LogReg test=0.717
p= 500 | KNN test=0.517 | LogReg test=0.617
p=1000 | KNN test=0.483 | LogReg test=0.633
p=5000 | KNN test=0.550 | LogReg test=0.650


# Lista 2